# Track B — YOLOv8n -> ONNX -> TensorRT FP16

Pipeline de entrenamiento para clasificación de residuos (`glass` / `paper` / `plastic`) sobre **Jetson Nano B01**.

**Target de inferencia (fijo, no negociable)**

| Componente | Versión |
|---|---|
| Jetson Nano | B01 (Maxwell sm_53, 128 CUDA cores, 4 GB unified RAM) |
| JetPack | 4.6.1 (L4T R32.7.1) |
| TensorRT | 8.2.1.8 |
| CUDA | 10.2.300 |
| cuDNN | 8.2.1 |
| Python (Nano) | 3.6.9 |
| OpenCV | 4.1.1 |

**Target de entrenamiento**

- Modo principal: instancia Vast.ai con RTX 4090 (`vastai/base-image:cuda-12.4.1-cudnn-devel-ubuntu22.04-py310`).
- Modo desarrollo: host local con cualquier GPU NVIDIA CUDA-capable.
- Stack se administra con `uv` (resuelto por `bootstrap.sh` antes de abrir el notebook).

**Modo de ejecución**

Headless vía `jupyter nbconvert --execute --inplace` dentro de `tmux`. La sesión sobrevive desconexión SSH; no es necesario mantener una pestaña abierta.

**Persistencia**

`huggingface_hub.CommitScheduler` empuja `runs/`, `manifests/` y `exports/` al repo privado `mitgar14/embebidos-3-models` cada 10 minutos. Un signal handler garantiza un último commit antes de que la instancia Vast.ai sea destruida.

**Pre-requisitos**

1. `bootstrap.sh` ejecutado en el host (instala `uv`, crea `/opt/venv/trackb`, registra el kernel `trackb`, monta cron watchdog auto-destroy).
2. Variables de entorno presentes: `HF_TOKEN`, `ROBOFLOW_API_KEY`.
3. Kernel `trackb` seleccionado (este notebook lo requiere).

**Pipeline (de arriba abajo)**

1. Detección de entorno + recursos.
2. Configuración + validación de tokens.
3. Workdir setup.
4. Verificación de stack.
5. Pre-flight Vast.ai (logind, tmux, cron).
6. Descarga del dataset Roboflow (`embebidos3/waste-3class-lwld8` v1-B, Fit-black 416×416).
7. Validación + visualización del dataset.
8. `CommitScheduler` + signal handlers.
9. Heartbeat watchdog (TRAINCHECK-style).
10. Entrenamiento YOLOv8n 416×416.
11. Evaluación split `test`.
12. Export ONNX `opset=11` (FP16 se aplica al compilar el engine en el Nano).
13. Gate 4 — validación opcional con Polygraphy (Docker NGC).
14. Manifest JSON + commit final + auto-destroy.


## 1. Detección de entorno

Identificamos si estamos en Vast.ai (`VAST_CONTAINERLABEL` presente) o local. Reportamos GPU, RAM y disco. En Vast.ai capturamos `VAST_CONTAINERLABEL` (formato `C.XXXXXXX`) para el `vastai destroy` final.


In [ ]:
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

IS_VASTAI = "VAST_CONTAINERLABEL" in os.environ
VAST_CONTAINERLABEL = os.environ.get("VAST_CONTAINERLABEL", "")

print(f"Host          : {platform.node()}")
print(f"Python        : {platform.python_version()}")
print(f"Plataforma    : {platform.platform()}")
print(f"IS_VASTAI     : {IS_VASTAI}")
if IS_VASTAI:
    print(f"Vast label    : {VAST_CONTAINERLABEL}")

try:
    r = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
         "--format=csv,noheader"],
        capture_output=True, text=True, timeout=5, check=True,
    )
    print(f"GPU           : {r.stdout.strip()}")
except Exception as e:
    print(f"GPU           : nvidia-smi indisponible ({e})")

try:
    with open("/proc/meminfo") as f:
        for line in f:
            if line.startswith("MemTotal"):
                ram_gb = int(line.split()[1]) / 1024 / 1024
                print(f"RAM total     : {ram_gb:.1f} GB")
                break
except FileNotFoundError:
    pass

du = shutil.disk_usage("/")
print(f"Disco libre   : {du.free/1e9:.1f} GB de {du.total/1e9:.1f} GB en /")


## 2. Configuración global y tokens

Tokens (`HF_TOKEN`, `ROBOFLOW_API_KEY`) se leen del entorno. Si faltan, el notebook aborta antes de descargar nada.

`CommitScheduler` apunta al repo HF `mitgar14/embebidos-3-models` (privado). Roboflow workspace `embebidos3`, proyecto `waste-3class-lwld8`, versión 1 (`v1-B`, Fit-black 416×416). Batch autoscale: 64 en Vast.ai (RTX 4090, 24 GB), 16 en local (asume GPU mid-range).


In [ ]:
CFG = {
    "rf_workspace": "embebidos3",
    "rf_project": "waste-3class-lwld8",
    "rf_version": 1,
    "rf_format": "yolov8",
    "hf_repo": "mitgar14/embebidos-3-models",
    "model_arch": "yolov8n",
    "imgsz": 416,
    "epochs": 100,
    "batch": 64 if IS_VASTAI else 16,
    "patience": 20,
    "device": 0,
    "seed": 42,
    "project_root": "/workspace/embebidos-3" if IS_VASTAI else os.path.expanduser("~/embebidos-3"),
    "commit_every_min": 10,
}

HF_TOKEN = os.environ.get("HF_TOKEN")
RF_KEY = os.environ.get("ROBOFLOW_API_KEY")
missing = [k for k, v in {"HF_TOKEN": HF_TOKEN, "ROBOFLOW_API_KEY": RF_KEY}.items() if not v]
if missing:
    raise EnvironmentError(f"Variables de entorno faltantes: {missing}. Exporta antes de ejecutar.")

print("Config:")
for k, v in CFG.items():
    print(f"  {k:18s}= {v}")
print(f"\nHF_TOKEN              = ****{HF_TOKEN[-4:]}")
print(f"ROBOFLOW_API_KEY      = ****{RF_KEY[-4:]}")


## 3. Directorio de trabajo

Vast.ai monta `/workspace` como volumen persistente del contenedor; local cae a `~/embebidos-3`. Sub-directorios:

- `datasets/` — datasets de Roboflow.
- `runs/` — outputs de YOLO (`detect/train/...`, `detect/val_test/...`) + `heartbeat.jsonl`.
- `manifests/` — JSON consolidados (`eval_summary.json`, `gate3_onnx.json`, `gate4_polygraphy.json`, `manifest.json`).
- `exports/` — `best.onnx` listo para transferir al Nano.


In [ ]:
ROOT = Path(CFG["project_root"])
DATASETS = ROOT / "datasets"
RUNS = ROOT / "runs"
MANIFESTS = ROOT / "manifests"
EXPORTS = ROOT / "exports"

for p in (ROOT, DATASETS, RUNS, MANIFESTS, EXPORTS):
    p.mkdir(parents=True, exist_ok=True)

os.chdir(ROOT)
print(f"CWD       : {Path.cwd()}")
print(f"DATASETS  : {DATASETS}")
print(f"RUNS      : {RUNS}")
print(f"MANIFESTS : {MANIFESTS}")
print(f"EXPORTS   : {EXPORTS}")


## 4. Verificación de stack

`bootstrap.sh` ya instaló el stack en `/opt/venv/trackb` (Vast.ai) o en el `.venv` local. Aquí solo verificamos versiones; si algún paquete falta lo agregamos con `uv pip install --python <venv>/bin/python ...`.

**Constraints duros**

| Paquete | Versión | Razón |
|---|---|---|
| `numpy` | `<2.0` | YOLO + TRT no compatibles con NumPy 2.x en este entorno. |
| `ultralytics` | `>=8.4.46,<8.5` | Rama de export ONNX estable; INT8 calib fix incluido. |
| `onnxslim` | `>=0.1.34` | Reemplaza `onnxsim` en Ultralytics ≥8.3 (cambio interno upstream). |


In [ ]:
import importlib

PY = "/opt/venv/trackb/bin/python" if IS_VASTAI else sys.executable

REQUIRED = {
    "torch": None,
    "torchvision": None,
    "ultralytics": ">=8.4.46,<8.5",
    "onnx": None,
    "onnxslim": ">=0.1.34",
    "onnxruntime": None,
    "numpy": "<2",
    "roboflow": None,
    "huggingface_hub": None,
    "matplotlib": None,
    "pandas": None,
    "PIL": None,
    "yaml": None,
}

_PIP_NAMES = {"PIL": "pillow", "yaml": "pyyaml"}

def _imported_version(name: str) -> str | None:
    try:
        mod = importlib.import_module(name)
        return getattr(mod, "__version__", "unknown")
    except ImportError:
        return None

missing_pkgs: list[str] = []
for pkg, spec in REQUIRED.items():
    v = _imported_version(pkg)
    if v is None:
        missing_pkgs.append(pkg)
        print(f"  [FALTA] {pkg}")
    else:
        print(f"  [OK]    {pkg:18s}= {v}")

if missing_pkgs:
    install_args = [
        f"{_PIP_NAMES.get(p, p)}{REQUIRED[p] or ''}"
        for p in missing_pkgs
    ]
    print(f"\nInstalando vía uv: {install_args}")
    subprocess.run(
        ["uv", "pip", "install", "--python", PY, *install_args],
        check=True,
    )

import torch
import numpy as np
print(f"\ntorch          = {torch.__version__}")
print(f"cuda disponible = {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"device 0       = {torch.cuda.get_device_name(0)}")
    print(f"total_memory   = {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    raise RuntimeError("CUDA no disponible — entrenamiento abortado.")

if int(np.__version__.split(".")[0]) >= 2:
    raise RuntimeError(f"numpy {np.__version__} incompatible (requiere <2).")
print(f"numpy          = {np.__version__} (constraint <2 OK)")


## 5. Pre-flight Vast.ai

Solo se ejecuta en Vast.ai. Verifica:

- `/etc/systemd/logind.conf` declara `KillUserProcesses=no`. Sin esto, `tmux` muere al cerrar la sesión SSH y el entrenamiento se interrumpe.
- Hay al menos una sesión `tmux` activa. Recomendación: lanzar `jupyter nbconvert --execute --inplace --ExecutePreprocessor.kernel_name=trackb notebooks/train_track_b_yolov8.ipynb` dentro de `tmux new -s training`.
- El cron watchdog `/opt/scripts/check-gpu-idle.sh` existe (instalado por `bootstrap.sh`; destruye la instancia si GPU < 5% durante 30 min).


In [ ]:
if IS_VASTAI:
    try:
        with open("/etc/systemd/logind.conf") as f:
            content = f.read()
        if "KillUserProcesses=no" in content:
            print("[OK]   KillUserProcesses=no presente.")
        else:
            print("[WARN] KillUserProcesses=no no encontrado — tmux puede morir tras logout.")
    except FileNotFoundError:
        print("[WARN] logind.conf no encontrado.")

    r = subprocess.run(["tmux", "ls"], capture_output=True, text=True)
    if r.returncode == 0 and r.stdout.strip():
        print(f"[OK]   tmux sessions:\n{r.stdout.rstrip()}")
    else:
        print("[WARN] No hay sesiones tmux activas.")

    watchdog = Path("/opt/scripts/check-gpu-idle.sh")
    if watchdog.exists():
        print(f"[OK]   cron watchdog presente: {watchdog}")
    else:
        print("[WARN] cron watchdog ausente — auto-destroy por idle no activo.")
else:
    print("Local — pre-flight Vast.ai omitido.")


## 6. Descarga del dataset (Roboflow)

Workspace `embebidos3`, proyecto `waste-3class-lwld8`, versión 1 (`v1-B`, Fit-black 416×416). Tres clases: `glass`, `paper`, `plastic`.

El SDK de Roboflow (≤1.3.9) tiene un bug intermitente al resolver `location=`. Aplicamos una cascada de tres estrategias antes de fallar:

1. `version.download("yolov8", location=str(target))` — happy path.
2. `version.download("yolov8")` y mover el resultado a `target`.
3. Re-instanciar `Project` y reintentar (3 retries con backoff exponencial).

**Importante**: NO se setea la env var `DATASET_DIRECTORY`. Entra en conflicto con el argumento `location` y deja ambas ubicaciones vacías.


In [ ]:
import time
import shutil as _shutil
from roboflow import Roboflow

if "DATASET_DIRECTORY" in os.environ:
    del os.environ["DATASET_DIRECTORY"]

DATA_DIR = DATASETS / f"{CFG['rf_project']}-v{CFG['rf_version']}"

def _download_rf(target: Path) -> Path:
    rf = Roboflow(api_key=RF_KEY)
    ver = rf.workspace(CFG["rf_workspace"]).project(CFG["rf_project"]).version(CFG["rf_version"])

    try:
        ds = ver.download(CFG["rf_format"], location=str(target), overwrite=False)
        return Path(ds.location)
    except Exception as e:
        print(f"  intento 1 (location) falló: {e}")

    try:
        ds = ver.download(CFG["rf_format"])
        src = Path(ds.location)
        if target.exists():
            _shutil.rmtree(target)
        _shutil.move(str(src), str(target))
        return target
    except Exception as e:
        print(f"  intento 2 (move) falló: {e}")

    for attempt in range(3):
        time.sleep(2 ** attempt)
        try:
            rf2 = Roboflow(api_key=RF_KEY)
            ver2 = rf2.workspace(CFG["rf_workspace"]).project(CFG["rf_project"]).version(CFG["rf_version"])
            ds = ver2.download(CFG["rf_format"], location=str(target), overwrite=False)
            return Path(ds.location)
        except Exception as e:
            print(f"  retry {attempt+1}/3 falló: {e}")

    raise RuntimeError("Roboflow download falló tras 3 estrategias.")

if (DATA_DIR / "data.yaml").exists():
    print(f"[skip] dataset ya presente en {DATA_DIR}")
else:
    DATA_DIR.parent.mkdir(parents=True, exist_ok=True)
    loc = _download_rf(DATA_DIR)
    print(f"Dataset descargado en {loc}")

DATA_YAML = DATA_DIR / "data.yaml"
if not DATA_YAML.exists():
    # Fallback: localizar data.yaml en cualquier subdirectorio
    found = next(DATA_DIR.rglob("data.yaml"), None)
    if found is None:
        raise FileNotFoundError(f"data.yaml no encontrado bajo {DATA_DIR}.")
    DATA_YAML = found
print(f"data.yaml resuelto: {DATA_YAML}")


## 7. Validación + visualización del dataset

Conteo `train` / `valid` / `test`, distribución de clases (barplot) y mosaico de 30 imágenes aleatorias del split `train` para inspección visual rápida.


In [ ]:
import random
import yaml
import matplotlib.pyplot as plt
from PIL import Image

with open(DATA_YAML) as f:
    dy = yaml.safe_load(f)

raw_names = dy.get("names", [])
if isinstance(raw_names, dict):
    CLASSES = [raw_names[k] for k in sorted(raw_names)]
else:
    CLASSES = list(raw_names)
print(f"Clases ({len(CLASSES)}): {CLASSES}")

def _count_split(split: str) -> int:
    p = DATA_DIR / split / "images"
    if not p.exists():
        # Roboflow yolov8 usa 'valid' (no 'val') — pero soportamos ambos
        alt = DATA_DIR / ("valid" if split == "val" else split) / "images"
        if alt.exists():
            return sum(1 for _ in alt.glob("*.*"))
        return 0
    return sum(1 for _ in p.glob("*.*"))

for s in ("train", "valid", "test"):
    print(f"  {s:6s}: {_count_split(s):4d}")

class_counts = {c: 0 for c in CLASSES}
labels_dir = DATA_DIR / "train" / "labels"
if labels_dir.exists():
    for lf in labels_dir.glob("*.txt"):
        with open(lf) as f:
            for line in f:
                parts = line.split()
                if parts:
                    idx = int(parts[0])
                    if 0 <= idx < len(CLASSES):
                        class_counts[CLASSES[idx]] += 1

fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(class_counts.keys(), class_counts.values())
ax.set_title("Distribución de clases — train")
ax.set_ylabel("Bounding boxes")
plt.tight_layout()
plt.show()
print(f"Bbox por clase: {class_counts}")

train_imgs = list((DATA_DIR / "train" / "images").glob("*.*"))
if train_imgs:
    random.seed(CFG["seed"])
    sample = random.sample(train_imgs, min(30, len(train_imgs)))
    rows, cols = 5, 6
    fig, axs = plt.subplots(rows, cols, figsize=(15, 12))
    for ax, p in zip(axs.flat, sample):
        ax.imshow(Image.open(p))
        ax.set_title(p.stem[:20], fontsize=7)
        ax.axis("off")
    for ax in axs.flat[len(sample):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()


## 8. Persistencia incremental — `CommitScheduler` + signal handlers

`CommitScheduler` sincroniza `runs/`, `manifests/` y `exports/` con el repo HF cada `commit_every_min` minutos. `squash_history=True` mantiene el repo ligero (solo el último estado).

**Signal chain**

Vast.ai envía `SIGTERM` cuando destruye la instancia. Capturamos `SIGTERM` y `SIGINT`, lanzamos `SystemExit(0)`, y la función registrada con `atexit` fuerza un último `trigger().result(timeout=180) + stop()` antes de salir. Esto garantiza que el `best.onnx` y el `manifest.json` queden en el Hub aunque la instancia muera abruptamente.


In [ ]:
import atexit
import signal
from huggingface_hub import CommitScheduler, HfApi

HF_API = HfApi(token=HF_TOKEN)
HF_API.create_repo(repo_id=CFG["hf_repo"], private=True, exist_ok=True)

scheduler = CommitScheduler(
    repo_id=CFG["hf_repo"],
    repo_type="model",
    folder_path=str(ROOT),
    every=CFG["commit_every_min"],
    squash_history=True,
    allow_patterns=["runs/**", "manifests/**", "exports/**"],
    ignore_patterns=["*.cache", "*.tmp", "__pycache__/**", "*.pyc"],
    token=HF_TOKEN,
)
print(f"CommitScheduler activo: cada {CFG['commit_every_min']} min -> {CFG['hf_repo']}")

_final_done = False

def _final_commit() -> None:
    global _final_done
    if _final_done:
        return
    _final_done = True
    print("[atexit] Forzando commit final...")
    try:
        scheduler.trigger().result(timeout=180)
        scheduler.stop()
        print("[atexit] Commit final OK.")
    except Exception as e:
        print(f"[atexit] Commit final falló: {e}")

atexit.register(_final_commit)

def _sig_handler(signum, frame):
    print(f"[signal] {signum} recibido — saliendo limpio.")
    raise SystemExit(0)

signal.signal(signal.SIGTERM, _sig_handler)
signal.signal(signal.SIGINT, _sig_handler)
print("Signal handlers SIGTERM/SIGINT registrados.")


## 9. Heartbeat watchdog (TRAINCHECK-style)

Hilo daemon que escribe `runs/heartbeat.jsonl` cada 30 s. Cada línea incluye:

- `ts` (ISO UTC), `epoch`, `total_epochs`, `loss`, `grad_norm`, `lr`.
- `gpu_mem_mb`, `elapsed_s`, `eta_s`.

El estado se actualiza desde un callback `on_train_epoch_end` registrado en Ultralytics. Patrón inspirado en TRAINCHECK (arXiv:2506.14813) — permite detectar silent crashes y stalls inspeccionando el JSONL en HF Hub.


In [ ]:
import json
import threading
import time as _time
from datetime import datetime, timezone

HEARTBEAT_PATH = RUNS / "heartbeat.jsonl"
HEARTBEAT_PATH.parent.mkdir(parents=True, exist_ok=True)

hb_state = {
    "epoch": 0,
    "total_epochs": CFG["epochs"],
    "loss": None,
    "grad_norm": None,
    "lr": None,
    "started_at": _time.time(),
}
hb_stop = threading.Event()

def _gpu_mem_mb() -> float:
    if not torch.cuda.is_available():
        return 0.0
    return torch.cuda.memory_allocated(0) / 1024 / 1024

def _heartbeat_loop() -> None:
    while not hb_stop.is_set():
        elapsed = _time.time() - hb_state["started_at"]
        ep = hb_state["epoch"]
        eta = (elapsed / ep * (hb_state["total_epochs"] - ep)) if ep > 0 else None
        rec = {
            "ts": datetime.now(timezone.utc).isoformat(),
            "epoch": ep,
            "total_epochs": hb_state["total_epochs"],
            "loss": hb_state["loss"],
            "grad_norm": hb_state["grad_norm"],
            "lr": hb_state["lr"],
            "gpu_mem_mb": _gpu_mem_mb(),
            "elapsed_s": elapsed,
            "eta_s": eta,
        }
        with open(HEARTBEAT_PATH, "a") as f:
            f.write(json.dumps(rec) + "\n")
        hb_stop.wait(30)

hb_thread = threading.Thread(target=_heartbeat_loop, daemon=True, name="heartbeat")
hb_thread.start()
print(f"Heartbeat activo -> {HEARTBEAT_PATH}")


## 10. Entrenamiento — YOLOv8n 416×416

Modelo base `yolov8n.pt` (preentrenado COCO), fine-tuning sobre las 3 clases. Augmentation por defecto de Ultralytics (sin sobreajustar). Callback `on_train_epoch_end` actualiza `hb_state` con `loss`, `grad_norm` y `lr` para que el heartbeat los registre.

Outputs: `runs/detect/train/weights/best.pt`, `runs/detect/train/results.csv`.


In [ ]:
from ultralytics import YOLO

def _grad_norm(model) -> float:
    total = 0.0
    for p in model.parameters():
        if p.grad is not None:
            total += p.grad.detach().data.norm(2).item() ** 2
    return total ** 0.5

def _on_train_epoch_end(trainer) -> None:
    hb_state["epoch"] = int(trainer.epoch) + 1
    if getattr(trainer, "loss", None) is not None:
        try:
            hb_state["loss"] = float(trainer.loss.detach().item())
        except Exception:
            pass
    try:
        hb_state["grad_norm"] = _grad_norm(trainer.model)
    except Exception:
        pass
    try:
        opt = trainer.optimizer
        if opt and opt.param_groups:
            hb_state["lr"] = float(opt.param_groups[0]["lr"])
    except Exception:
        pass

model = YOLO(f"{CFG['model_arch']}.pt")
model.add_callback("on_train_epoch_end", _on_train_epoch_end)

results = model.train(
    data=str(DATA_YAML),
    epochs=CFG["epochs"],
    imgsz=CFG["imgsz"],
    batch=CFG["batch"],
    patience=CFG["patience"],
    device=CFG["device"],
    seed=CFG["seed"],
    project=str(RUNS / "detect"),
    name="train",
    exist_ok=True,
    plots=True,
    verbose=True,
)

BEST_PT = RUNS / "detect" / "train" / "weights" / "best.pt"
print(f"\nbest.pt -> {BEST_PT} (exists={BEST_PT.exists()})")


## 11. Evaluación sobre el split `test`

mAP@0.5, mAP@0.5:0.95, precision y recall por clase. Se guarda `manifests/eval_summary.json` para que el `CommitScheduler` lo suba a HF Hub en el siguiente tick.


In [ ]:
best_model = YOLO(str(BEST_PT))
metrics = best_model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=CFG["imgsz"],
    batch=CFG["batch"],
    device=CFG["device"],
    project=str(RUNS / "detect"),
    name="val_test",
    exist_ok=True,
    plots=True,
    verbose=True,
)

eval_summary = {
    "mAP50": float(metrics.box.map50),
    "mAP50_95": float(metrics.box.map),
    "precision_mean": float(metrics.box.mp),
    "recall_mean": float(metrics.box.mr),
    "per_class": {
        CLASSES[i]: {
            "precision": float(metrics.box.p[i]),
            "recall": float(metrics.box.r[i]),
            "AP50": float(metrics.box.ap50[i]),
            "AP50_95": float(metrics.box.ap[i]),
        }
        for i in range(len(CLASSES))
    },
}

EVAL_JSON = MANIFESTS / "eval_summary.json"
with open(EVAL_JSON, "w") as f:
    json.dump(eval_summary, f, indent=2)
print(json.dumps(eval_summary, indent=2))


## 12. Export ONNX — `opset=11`, `nms=False`

**Opset 11** es el más alto que TensorRT 8.2.1.8 del JetPack 4.6.1 acepta de forma estable (cap interna en 13, dejamos margen).

**NMS desacoplado del grafo (`nms=False`)** — Maxwell sm_53 + TRT 8.2 tiene tres caminos posibles, en este orden de preferencia:

| Versión | Estrategia | Estado |
|---|---|---|
| **V0 (default)** | `cv2.dnn.NMSBoxes` (CPU) en post-proceso Python | Compatibilidad garantizada. Penalización ~3-5 ms/frame. |
| **V1 (smoke test)** | `EfficientNMS_TRT` plugin embebido en el engine | Disponible en JP 4.6.1 binary (fix #1538 commit `3235cc2`). Validar funcionalmente. |
| **V2 (fallback)** | `BatchedNMSDynamic_TRT` plugin | Plugin estable, mismo binary. Si V1 falla, fallback inmediato. |

El engine TRT FP16 se compila **en el Nano** vía `trtexec --fp16 --workspace=1024 ...`. Aquí solo producimos el `.onnx`.

**Gate 3** — validamos en el `.onnx`: `opset==11`, `ir_version<=8`, y ningún op en la blacklist (`GridSample`, `MultiHeadAttention`, `RoiAlign`, `NonZero`, `Reciprocal`, etc.).


In [ ]:
import onnx

ONNX_BLACKLIST = {
    "GridSample", "MultiHeadAttention", "RoiAlign", "NonZero",
    "Reciprocal", "QLinearConv", "QLinearMatMul",
}

ONNX_PATH = EXPORTS / "best.onnx"

export_str = best_model.export(
    format="onnx",
    imgsz=CFG["imgsz"],
    opset=11,
    simplify=True,
    dynamic=False,
    nms=False,
    half=False,
    device="cpu",
)
export_path = Path(export_str)
print(f"Exportado por Ultralytics: {export_path}")
_shutil.copy(export_path, ONNX_PATH)
print(f"Copiado a: {ONNX_PATH} ({ONNX_PATH.stat().st_size/(1024**2):.2f} MB)")

mdl = onnx.load(str(ONNX_PATH))
onnx.checker.check_model(mdl)
opset_v = mdl.opset_import[0].version
ir_v = mdl.ir_version
ops = {n.op_type for n in mdl.graph.node}
forbidden = ops & ONNX_BLACKLIST
in_shape = [d.dim_value for d in mdl.graph.input[0].type.tensor_type.shape.dim]

gate3 = {
    "opset": int(opset_v),
    "ir_version": int(ir_v),
    "input_shape": in_shape,
    "n_outputs": len(mdl.graph.output),
    "ops_count": len(ops),
    "ops_forbidden": sorted(forbidden),
    "checker": "ok",
    "pass": opset_v == 11 and ir_v <= 8 and not forbidden,
}

GATE3_JSON = MANIFESTS / "gate3_onnx.json"
with open(GATE3_JSON, "w") as f:
    json.dump(gate3, f, indent=2)
print(json.dumps(gate3, indent=2))

if not gate3["pass"]:
    raise RuntimeError(f"Gate 3 falló: {gate3}")


## 13. Gate 4 — Polygraphy (opcional)

Si Docker está disponible, levantamos un contenedor NGC `nvcr.io/nvidia/tensorrt:21.11-py3` (TRT 8.0.3 / cuDNN 8.2 / CUDA 11.4 — la combinación NGC más cercana a JetPack 4.6.1) y corremos `polygraphy inspect model` sobre el `.onnx`. Esto valida que el grafo es parseable por una rama de TensorRT compatible antes de gastar 15-45 min compilando el engine en el Nano.

Si Docker no está disponible (Vast.ai container o entorno restringido), se marca como `skipped` sin romper el pipeline.


In [ ]:
def _has_docker() -> bool:
    try:
        r = subprocess.run(["docker", "--version"], capture_output=True, text=True, timeout=3)
        return r.returncode == 0
    except (FileNotFoundError, subprocess.TimeoutExpired):
        return False

GATE4_JSON = MANIFESTS / "gate4_polygraphy.json"

if _has_docker():
    cmd = [
        "docker", "run", "--rm", "--gpus", "all",
        "-v", f"{EXPORTS}:/work",
        "nvcr.io/nvidia/tensorrt:21.11-py3",
        "polygraphy", "inspect", "model", "/work/best.onnx",
        "--mode=basic",
    ]
    r = subprocess.run(cmd, capture_output=True, text=True)
    gate4 = {
        "docker": True,
        "returncode": r.returncode,
        "stdout_tail": r.stdout[-1500:],
        "stderr_tail": r.stderr[-1500:],
        "pass": r.returncode == 0,
    }
else:
    gate4 = {"docker": False, "skipped": True, "reason": "docker no disponible", "pass": None}

with open(GATE4_JSON, "w") as f:
    json.dump(gate4, f, indent=2)
print(json.dumps(gate4, indent=2))


## 14. Manifest consolidado + commit final + auto-destroy

Construimos `manifest.json` con:

- Hashes SHA256 de `best.pt` y `best.onnx`.
- Métricas de evaluación.
- Resultados de Gate 3 y Gate 4.
- Target Nano (`jetpack: 4.6.1`, `l4t: R32.7.1`, `trt: 8.2.1.8`).
- Estrategia NMS V0/V1/V2.
- Versiones del stack de training y del host.

Tras escribirlo, detenemos el heartbeat, forzamos un último commit (`trigger().result()` + `stop()`) y — solo en Vast.ai — programamos `vastai destroy instance <id>` con 30 s de margen para que la última escritura al Hub se complete.


In [ ]:
import hashlib

def _sha256(p: Path) -> str | None:
    if not p.exists():
        return None
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 16), b""):
            h.update(chunk)
    return h.hexdigest()

manifest = {
    "model_arch": CFG["model_arch"],
    "imgsz": CFG["imgsz"],
    "classes": CLASSES,
    "dataset": {
        "workspace": CFG["rf_workspace"],
        "project": CFG["rf_project"],
        "version": CFG["rf_version"],
        "format": CFG["rf_format"],
        "data_yaml": str(DATA_YAML.relative_to(ROOT)),
    },
    "training": {
        "epochs_max": CFG["epochs"],
        "epochs_run": hb_state["epoch"],
        "batch": CFG["batch"],
        "patience": CFG["patience"],
        "seed": CFG["seed"],
    },
    "artifacts": {
        "best_pt": {
            "path": str(BEST_PT.relative_to(ROOT)),
            "sha256": _sha256(BEST_PT),
            "size_mb": round(BEST_PT.stat().st_size / (1024**2), 2) if BEST_PT.exists() else None,
        },
        "best_onnx": {
            "path": str(ONNX_PATH.relative_to(ROOT)),
            "sha256": _sha256(ONNX_PATH),
            "size_mb": round(ONNX_PATH.stat().st_size / (1024**2), 2) if ONNX_PATH.exists() else None,
            "opset": 11,
            "ir_version": gate3["ir_version"],
        },
    },
    "evaluation": eval_summary,
    "gates": {
        "gate3_onnx": gate3,
        "gate4_polygraphy": gate4,
    },
    "target_nano": {
        "device": "Jetson Nano B01",
        "jetpack": "4.6.1",
        "l4t": "R32.7.1",
        "cuda": "10.2.300",
        "cudnn": "8.2.1",
        "tensorrt": "8.2.1.8",
        "python": "3.6.9",
        "compute_capability": "5.3",
        "nms_strategy": {
            "v0_default": "cv2.dnn.NMSBoxes (CPU NumPy post-proc)",
            "v1_smoke_test": "EfficientNMS_TRT plugin embebido",
            "v2_fallback": "BatchedNMSDynamic_TRT plugin",
        },
        "trtexec": (
            f"trtexec --onnx=best.onnx --saveEngine=yolov8n_waste_fp16.engine "
            f"--fp16 --workspace=1024 --verbose"
        ),
    },
    "training_host": {
        "vastai": IS_VASTAI,
        "vast_label": VAST_CONTAINERLABEL if IS_VASTAI else None,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "python": platform.python_version(),
    },
    "stack": {
        "torch": torch.__version__,
        "ultralytics": _imported_version("ultralytics"),
        "onnx": _imported_version("onnx"),
        "onnxslim": _imported_version("onnxslim"),
        "onnxruntime": _imported_version("onnxruntime"),
        "numpy": np.__version__,
        "roboflow": _imported_version("roboflow"),
        "huggingface_hub": _imported_version("huggingface_hub"),
    },
    "timestamp": datetime.now(timezone.utc).isoformat(),
}

MANIFEST_JSON = MANIFESTS / "manifest.json"
with open(MANIFEST_JSON, "w") as f:
    json.dump(manifest, f, indent=2)
print(json.dumps(manifest, indent=2))

hb_stop.set()
hb_thread.join(timeout=5)
print("\nHeartbeat detenido.")

_final_commit()

if IS_VASTAI and VAST_CONTAINERLABEL:
    inst_id = VAST_CONTAINERLABEL.lstrip("C.")
    print(f"\n[auto-destroy] Programado para instancia {inst_id} en 30 s.")
    subprocess.Popen(
        ["bash", "-lc", f"sleep 30 && vastai destroy instance {inst_id}"],
        start_new_session=True,
    )
    print("[auto-destroy] Lanzado en background. Adiós.")
else:
    print("\nLocal — sin auto-destroy. Done.")
